# v20 簡易セレクタ v3: PF出力 + パラメータ + 井戸レジーム(parquet既存列) でスコア付け

(行 × PF)=1サンプル。特徴 =
  - そのPFの canonical 特徴 (pf_{k}_gr_* GR当てはまり, tdpf, ancc_std/delta)
  - 位置/合意距離 (pred_centered, rank, spread, md_since)
  - PFパラメータ (likelihood_scale, gr_power, init_pos_std, phys ...)
  - **井戸レジーム/局所(parquetの共有非PF列): eval_len, known_len, ktvt_std, slp_z, z, gr, ... ← 既にparquetにある**
ターゲット=|err|。5-fold GroupKFold by well。最小スコアPF採用(hard)/確率加重(soft)。
狙い: 井戸レジーム × PFパラメータ/GR当てはまり の交互作用で「この井戸ならこのPF」を学習。

In [1]:
import re, gc, sys, json, warnings, numpy as np, pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

PARQUET   = Path("C:/Users/kosaka256/Documents/rogii_claude/v20/artifacts_v20/data/train.parquet")
BANK_JSON = Path("C:/Users/kosaka256/Documents/rogii_claude/v19/artifacts_v19/pf_bank.json")
V9_DIR    = Path("C:/Users/kosaka256/Documents/rogii_claude/v9")
ROW_FRAC  = 0.2

df = pd.read_parquet(PARQUET)
if ROW_FRAC < 1.0:
    step = max(1, int(round(1.0 / ROW_FRAC)))
    df = df.iloc[::step].reset_index(drop=True)

pf_cols = sorted([c for c in df.columns if re.fullmatch(r"pf_ancc_\d+", c)], key=lambda c: int(c.split("_")[-1]))
SUF = [int(c.split("_")[-1]) for c in pf_cols]
K = len(pf_cols); N = len(df)
print("PF本数 K =", K, "| rows =", N, "| suffix =", SUF)

true_tvt = (df["last_known_tvt"].to_numpy(np.float64) + df["target"].to_numpy(np.float64))
preds = np.stack([df[c].to_numpy(np.float64) for c in pf_cols], 1)
err   = preds - true_tvt[:, None]
g     = df["well"].to_numpy()
md_since = df["md_since"].to_numpy(np.float32) if "md_since" in df.columns else np.zeros(N, np.float32)

def rmse(e): e = e[np.isfinite(e)]; return float(np.sqrt(np.mean(e**2)))
print("\n-- baseline (raw TVT) --")
for j, c in enumerate(pf_cols): print(f"  {c:12s} overall={rmse(err[:,j]):.3f}")
print(f"  mean-ensemble overall={rmse(preds.mean(1)-true_tvt):.3f}")
best_row = np.argmin(np.abs(err), 1)
print(f"  row-oracle    overall={rmse(err[np.arange(N), best_row]):.3f}")
well_err = []
for _, idx in pd.Series(np.arange(N)).groupby(g).groups.items():
    idx = np.array(idx); we = err[idx]; well_err.append(we[:, int(np.argmin(np.sqrt(np.mean(we**2, 0))))])
WELL_ORACLE = rmse(np.concatenate(well_err)); print(f"  well-oracle   overall={WELL_ORACLE:.3f}")

# ---- 共有(非PF)列の厳選サブセット = 井戸レジーム/局所(parquet既存)。全候補に共有 ----
SHARED_WANT = ["eval_len","known_len","slp_z","ktvt_range","ktvt_std","tw_range","spatial_knn_dist",
               "dense_std","form_std_d","beam_std_d","sig_range","sig_gr_best_abs","sig_gr_resid_std",
               "z","dz","dzdmd","dxdmd","dydmd","gr","gr_d1","gr_d2","gr_env","gr_nrg","gr_vs_tw_anc","last_known_tvt"]
SHARED = [c for c in SHARED_WANT if c in df.columns]
print(f"\n共有(井戸/局所)特徴 {len(SHARED)}:", SHARED)

PARAMS = [e["params"] for e in json.loads(BANK_JSON.read_text())]
if K > len(PARAMS):
    sys.path.append(str(V9_DIR)); from pf_gpu_run import PF_PARAM_SETS_RAW
    for raw in PF_PARAM_SETS_RAW.values():
        r = dict(raw); r["use_phys"] = False; r.setdefault("phys_sig", 1.0); PARAMS.append(r)
PARAMS = PARAMS[:K]
PKEYS = ["likelihood_scale","gr_power","init_pos_std","mom","resamp","phys_sig","n_particles","rate_clip","jump_prob"]
if len(PARAMS) != K: print("warn: param数!=K → パラメータ特徴スキップ"); PARAMS = None

PF本数 K = 7 | rows = 756798 | suffix = [1, 2, 3, 4, 5, 6, 7]

-- baseline (raw TVT) --
  pf_ancc_1    overall=9.485
  pf_ancc_2    overall=10.364
  pf_ancc_3    overall=9.473
  pf_ancc_4    overall=9.526
  pf_ancc_5    overall=9.485
  pf_ancc_6    overall=10.364
  pf_ancc_7    overall=9.473
  mean-ensemble overall=8.605
  row-oracle    overall=5.420
  well-oracle   overall=6.244

共有(井戸/局所)特徴 25: ['eval_len', 'known_len', 'slp_z', 'ktvt_range', 'ktvt_std', 'tw_range', 'spatial_knn_dist', 'dense_std', 'form_std_d', 'beam_std_d', 'sig_range', 'sig_gr_best_abs', 'sig_gr_resid_std', 'z', 'dz', 'dzdmd', 'dxdmd', 'dydmd', 'gr', 'gr_d1', 'gr_d2', 'gr_env', 'gr_nrg', 'gr_vs_tw_anc', 'last_known_tvt']


In [2]:
# ===== 各PFの canonical 特徴 + 位置 + パラメータ + 共有(井戸/局所) を long に =====
def block_map(k):
    out = {}; pk = f"pf_{k}_"
    for c in df.columns:
        if c == f"pf_ancc_std_{k}": out[c] = "ancc_std"
        elif c == f"pf_ancc_delta_{k}": out[c] = "ancc_delta"
        elif c.startswith(pk): out[c] = c[len(pk):]
        elif c.startswith("pf_tdpf") and c.endswith(f"_{k}"): out[c] = c[:-(len(str(k)) + 1)]
    return out
maps = [block_map(k) for k in SUF]
common = None
for m in maps:
    s = set(m.values()); common = s if common is None else (common & s)
common = sorted(common)
print(f"per-PF canonical 特徴: {len(common)} 本")

med = np.median(preds, 1); mean = preds.mean(1); spread = preds.std(1)
rank = np.argsort(np.argsort(preds, 1), 1) / max(K - 1, 1)
shared_arr = {c: df[c].to_numpy(np.float32) for c in SHARED}
blocks = []
for ki, k in enumerate(SUF):
    inv = {v: c for c, v in maps[ki].items()}
    d = {name: df[inv[name]].to_numpy(np.float32) for name in common}
    d["pred_centered"] = (preds[:, ki] - med).astype(np.float32)
    d["pred_minus_mean"] = (preds[:, ki] - mean).astype(np.float32)
    d["rank"] = rank[:, ki].astype(np.float32); d["spread"] = spread.astype(np.float32); d["md_since"] = md_since
    for c in SHARED: d[f"w_{c}"] = shared_arr[c]          # 井戸/局所(共有)
    if PARAMS is not None:
        for key in PKEYS: d[f"p_{key}"] = np.full(N, float(PARAMS[ki].get(key, 0.0)), np.float32)
        d["p_use_phys"] = np.full(N, 1.0 if PARAMS[ki].get("use_phys", True) else 0.0, np.float32)
    blocks.append(pd.DataFrame(d))
Xl = pd.concat(blocks, ignore_index=True); yl = np.abs(err).T.reshape(-1).astype(np.float32); gl = np.tile(g, K)
print("long:", Xl.shape); del blocks; gc.collect()

oof = np.zeros(len(Xl), np.float32)
for fi, (tr, va) in enumerate(GroupKFold(5).split(Xl, yl, groups=gl)):
    m = LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=127,
                      subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1)
    m.fit(Xl.iloc[tr], yl[tr]); oof[va] = m.predict(Xl.iloc[va]); print(f"fold {fi} done", flush=True); del m; gc.collect()

score = oof.reshape(K, N).T
sel = score.argmin(1); e_hard = err[np.arange(N), sel]
wsoft = np.exp(-score / (score.std() + 1e-6)); wsoft /= wsoft.sum(1, keepdims=True)
e_soft = (wsoft * preds).sum(1) - true_tvt
print("\n=== セレクタ結果 (raw TVT) ===")
print(f"  best-PF的中率 = {float((sel==best_row).mean()):.3f}  (ランダム={1/K:.3f})")
print(f"  mean-ensemble : {rmse(preds.mean(1)-true_tvt):.3f}")
print(f"  selector hard : {rmse(e_hard):.3f}")
print(f"  selector soft : {rmse(e_soft):.3f}")
print(f"  well-oracle   : {WELL_ORACLE:.3f}")
print(f"  row-oracle    : {rmse(err[np.arange(N), best_row]):.3f}")

per-PF canonical 特徴: 17 本
long: (5297586, 57)
fold 0 done
fold 1 done
fold 2 done
fold 3 done
fold 4 done

=== セレクタ結果 (raw TVT) ===
  best-PF的中率 = 0.185  (ランダム=0.143)
  mean-ensemble : 8.605
  selector hard : 9.080
  selector soft : 8.774
  well-oracle   : 6.244
  row-oracle    : 5.420
